# Atalaya · piloto acústico YAMNet

**Exploración técnica; no es un detector operativo.** El repositorio de
[DroneDetectionThesis v1.0.0](https://github.com/DroneDetectionThesis/Drone-detection-dataset/releases/tag/v1.0.0)
declara CC0, pero el artículo de sus autores (Svanström, Alonso-Fernandez y Englund,
[*A dataset for multi-sensor drone detection*](https://arxiv.org/pdf/2111.01888),
*Data in Brief* 39 (2021), 107521) indica que una parte pequeña del conjunto procede de YouTube
y que el audio se extrajo de los vídeos o se grabó aparte; la
[tesis del primer autor](https://hh.diva-portal.org/smash/get/diva2%3A1434532/FULLTEXT02.pdf)
sitúa 49 clips externos en el vídeo visible. No está documentado si algún WAV procede de ellos.

**Recuento de clips:** el inventario fijado contiene **90 WAV** (30 por clase), verificados por
SHA-256 en el preflight local de la data card. Este cuaderno usa una **muestra de 24**
(`N_PER_CLASS = 8` por clase, sin duplicados exactos) y verifica cada WAV contra ese inventario.
Las nuevas descargas y el entrenamiento quedan **bloqueados** hasta aclarar derechos por
archivo, revisar voz y completar el intake. Los audios se pueden escuchar abajo para esa revisión.

Pesos de [YAMNet](https://www.kaggle.com/models/google/yamnet) (Google, vía TF Hub/Kaggle):
licencia Apache 2.0 según la ficha oficial de Kaggle (véase la model card del piloto).

No hay IDs fiables de vuelo, sesión o lugar por archivo: los grupos son provisionales (un
clip, o un par de duplicados exactos). La partición train/validación/test por grupos es solo una
prueba de funcionamiento, con contaminación entre particiones no descartada, **no una medida de precisión en otro lugar ni de falsas alarmas de un
perímetro**. No se usa micrófono, audio de clientes, acceso al GitHub privado ni montaje de
Drive en el entorno de ejecución.

[Fraunhofer IDMT](https://zenodo.org/records/17525811) declara CC BY 4.0 y aporta una matriz
de ocho micrófonos para una prueba posterior de dirección de llegada. Una matriz única no
triangula posición por sí sola. [DADS](https://huggingface.co/datasets/geronimobasso/drone-audio-detection-samples)
queda excluido: pese a su etiqueta MIT, mezcla fuentes con restricciones de uso en seguridad,
defensa y uso comercial sin trazabilidad por muestra.


In [ ]:
# Dependencias para el entorno actual de Colab; se comprueba la versión de TensorFlow.
import importlib.metadata as meta
import subprocess, sys
tf_version = meta.version("tensorflow")
tf_minor = ".".join(tf_version.split(".")[:2])
print("TensorFlow de Colab:", tf_version)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "tensorflow-hub==0.16.1", f"tf-keras=={tf_minor}.*",
                       "librosa>=0.10.2,<0.12", "scikit-learn>=1.4,<2",
                       "soundfile>=0.12,<1"])
print("Dependencias preparadas. Si Colab pide reiniciar el entorno, reinícialo y vuelve a ejecutar.")


In [ ]:
import hashlib, json, random, urllib.request, wave
from pathlib import Path
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import tensorflow as tf
import tensorflow_hub as hub
from IPython.display import Audio, display
# Las funciones split_indices_by_group, choose_threshold, binary_metrics, group_bootstrap,
# false_alarms_per_hour y metric_caption se definen en la celda siguiente (código probado).

PILOT = json.loads('''{"source_commit":"e7a6eaf9760fa3f9ecc37d09cd11a8cd20be90dd","license_sha256":"a2010f343487d3f7618affe54f789f5487602331c0a8d03f49e9a7c547cf0499","group_status":"provisional","clips":[{"name":"BACKGROUND_001.wav","sha256":"e8790309a7888c58c1d1f0f3bfa7850f193af5f23fceb8fdc830aad610ffeb28","bytes":1764052,"group_id":"BACKGROUND_001","training_excluded":false},{"name":"BACKGROUND_002.wav","sha256":"3bdef3845531382240df338a41e555d42023d98e8db63c54380fab82c23bde39","bytes":1764048,"group_id":"BACKGROUND_002","training_excluded":false},{"name":"BACKGROUND_003.wav","sha256":"7fcad6fffda1d56ba8c236480b04289e8c079ff518c0770770c0cb8c09c43e95","bytes":1764052,"group_id":"BACKGROUND_003","training_excluded":false},{"name":"BACKGROUND_004.wav","sha256":"8db1c6e9977126387f5ab00fe5d09cbda1a91e12d315a59356059d606eab0552","bytes":1764048,"group_id":"BACKGROUND_004","training_excluded":false},{"name":"BACKGROUND_005.wav","sha256":"1e86479957e5ef21e5ab4c4b25d3df1f09647a34957e621790f199cac8dfac28","bytes":1764048,"group_id":"BACKGROUND_005","training_excluded":false},{"name":"BACKGROUND_006.wav","sha256":"f73498b347bd77f4824cd42887ef72c08eb928e79ad2d2dba54f105d5b23fff5","bytes":1764052,"group_id":"BACKGROUND_006","training_excluded":false},{"name":"BACKGROUND_007.wav","sha256":"737b05e544fce41d5d2f2e608bae5fd199a6695a80f4c31d2e6945490f15c91d","bytes":1764048,"group_id":"BACKGROUND_007","training_excluded":false},{"name":"BACKGROUND_008.wav","sha256":"7f03a35e12c520b95eeab235ce7bec4d174b5adbf506e398da1f9d8a2f3f51a4","bytes":1764048,"group_id":"BACKGROUND_008","training_excluded":false},{"name":"BACKGROUND_009.wav","sha256":"e3cb52f2d2abef93cfe80724de76a96f4de8fa05597139e0ce5ad7dd32db5d7d","bytes":1764048,"group_id":"BACKGROUND_009","training_excluded":false},{"name":"BACKGROUND_010.wav","sha256":"37200cb91d6f12b4e5e6c79cfe42576252f033c67e862f737d7147ff52981318","bytes":1764048,"group_id":"BACKGROUND_010","training_excluded":false},{"name":"BACKGROUND_011.wav","sha256":"c59c17282d104ca1383efbe9addff87dd3c0669fd0a3edea6923f4fe86a4f9cc","bytes":1764052,"group_id":"BACKGROUND_011","training_excluded":false},{"name":"BACKGROUND_012.wav","sha256":"9dc31639da818626ff0f0995c793f1d36c12450ef5fd7e275dfc4fa9e52fe1f9","bytes":1764048,"group_id":"BACKGROUND_012","training_excluded":false},{"name":"BACKGROUND_013.wav","sha256":"ed96b240e24ad16fca1c430cccbed95754354567c9c1b4e707772504213864a7","bytes":1764048,"group_id":"BACKGROUND_013","training_excluded":false},{"name":"BACKGROUND_014.wav","sha256":"9bc49486d332ea8dc745fb6ec0c6cf46d5c20ebcb2bd43324d06996bd22fb680","bytes":1764048,"group_id":"BACKGROUND_014","training_excluded":false},{"name":"BACKGROUND_015.wav","sha256":"37c9ca48639871650a6610f43a08f78574c2cabb02ff69d84fcd4e6b9863c01c","bytes":1764052,"group_id":"BACKGROUND_015","training_excluded":false},{"name":"BACKGROUND_016.wav","sha256":"ed26ab3e21d613b3b2400e1fd93cb81435cca26e4ae02283460ada3ebcce754f","bytes":1764048,"group_id":"BACKGROUND_016","training_excluded":false},{"name":"BACKGROUND_017.wav","sha256":"c8e44c10e29e7f10452e1bfa228ceffee1ade9aaefd0e7f4e18a1f801af2b647","bytes":1764048,"group_id":"BACKGROUND_017","training_excluded":false},{"name":"BACKGROUND_018.wav","sha256":"0d618c8b488c2463de551890ed92499e56a2e203306e7d92ffef7ce20a409d69","bytes":1764052,"group_id":"BACKGROUND_018","training_excluded":false},{"name":"BACKGROUND_019.wav","sha256":"37200cb91d6f12b4e5e6c79cfe42576252f033c67e862f737d7147ff52981318","bytes":1764048,"group_id":"BACKGROUND_010","training_excluded":true},{"name":"BACKGROUND_020.wav","sha256":"d1bdd8d0fffa1e99b2a672160456e1eb2bd8c2b4ed41c99320dd982d576bd637","bytes":1764052,"group_id":"BACKGROUND_020","training_excluded":false},{"name":"BACKGROUND_021.wav","sha256":"88c517a8e97b1735716e2257ae5cd55f355ed5834f427c3746918e1b9411e778","bytes":1764052,"group_id":"BACKGROUND_021","training_excluded":false},{"name":"BACKGROUND_022.wav","sha256":"33537f39d3143b27d5a9de3c66e0f1e00c599b73b263989814ff9f17e59ec4f9","bytes":1764052,"group_id":"BACKGROUND_022","training_excluded":false},{"name":"BACKGROUND_023.wav","sha256":"2fce1d9350ab2b453676e156000ff292a520ba710bbc571e69e7b19aa51ebe4f","bytes":1764048,"group_id":"BACKGROUND_023","training_excluded":false},{"name":"BACKGROUND_024.wav","sha256":"991877ffaa560d376332a878afdeb1c95c080aee7e2a5d1ff0f7da44ecc93ffa","bytes":1764048,"group_id":"BACKGROUND_024","training_excluded":false},{"name":"BACKGROUND_025.wav","sha256":"c441558280f77504792a2a0c6a8829422a7763b366a45aab18ac381c7c43f066","bytes":1764048,"group_id":"BACKGROUND_025","training_excluded":false},{"name":"BACKGROUND_026.wav","sha256":"55654455627d97f252f30f63ea7f37e5ce7d9a6c076eb3580c630bc46e98a6e9","bytes":1764052,"group_id":"BACKGROUND_026","training_excluded":false},{"name":"BACKGROUND_027.wav","sha256":"9f4edfaca8194afbb264120cb002f7e9d4bceb03f0fc2ec39a88d6af2b251cd5","bytes":1764048,"group_id":"BACKGROUND_027","training_excluded":false},{"name":"BACKGROUND_028.wav","sha256":"af35ea6d7947b5d520054067233f932637f3852f1dc0f6e3e9479e525d9eac3a","bytes":1764048,"group_id":"BACKGROUND_028","training_excluded":false},{"name":"BACKGROUND_029.wav","sha256":"785c4bd25ef23a1a9f66af55accfb3bce6a06149c79bf9d236b8ce042562f397","bytes":1764052,"group_id":"BACKGROUND_029","training_excluded":false},{"name":"BACKGROUND_030.wav","sha256":"34f988beed58747337bf53bed8be7fcc3f0a6aadb06d49dc486fc48ef97d6b71","bytes":1764052,"group_id":"BACKGROUND_030","training_excluded":false},{"name":"DRONE_001.wav","sha256":"3c3fe29253e442cee58bb02b064e590e060d3e3e1c4039d8eb05d2da3b2c368c","bytes":1764048,"group_id":"DRONE_001","training_excluded":false},{"name":"DRONE_002.wav","sha256":"a18e7825b0da4df748bea37d9392d92512192a7fe6447bea679d8980c8cfe3c4","bytes":1764052,"group_id":"DRONE_002","training_excluded":false},{"name":"DRONE_003.wav","sha256":"12f04588de940bcd15ae955eb1b9a8a250986cf129f5c481f64598cd32cd302b","bytes":1764048,"group_id":"DRONE_003","training_excluded":false},{"name":"DRONE_004.wav","sha256":"37c5062e4f07c9e05a67f7eb92a02785bddbf2fa10da8fd8596526755b54ec82","bytes":1764048,"group_id":"DRONE_004","training_excluded":false},{"name":"DRONE_005.wav","sha256":"d56924abc9fe6ae53fae99e3eb5b72d2805c605581fa3e129eac8e578623705a","bytes":1764048,"group_id":"DRONE_005","training_excluded":false},{"name":"DRONE_006.wav","sha256":"2079a59ea51d1e3468c29302d3c0c443b365fa27fe7dc53ac57a5fbc6d6b1edf","bytes":1764048,"group_id":"DRONE_006","training_excluded":false},{"name":"DRONE_007.wav","sha256":"46bb93a90bec955129f6e2b2043d8d406fe93dedcd50a2a1519f82db1c59e420","bytes":1764048,"group_id":"DRONE_007","training_excluded":false},{"name":"DRONE_008.wav","sha256":"3001968764aafe76b604c455a7464f19dde8bd5b91fc24f1f486be6df5daa749","bytes":1764048,"group_id":"DRONE_008","training_excluded":false},{"name":"DRONE_009.wav","sha256":"b17bff5669ee90a509af546c0edf0ffb4703c26920b7137c71010edc9186f897","bytes":1764048,"group_id":"DRONE_009","training_excluded":false},{"name":"DRONE_010.wav","sha256":"4798cb55d03757b17b4f43f40f9173eec0a0205de9797cce5424420c6833859c","bytes":1764048,"group_id":"DRONE_010","training_excluded":false},{"name":"DRONE_011.wav","sha256":"d556d8297499b37f58ac90d0d3f865d553495688230905c52bd9c3019829cec9","bytes":1764048,"group_id":"DRONE_011","training_excluded":false},{"name":"DRONE_012.wav","sha256":"7631a3915d5ac73cc704c12fef7de1c6020c0181c04f58be90318ec0f870b2d1","bytes":1764048,"group_id":"DRONE_012","training_excluded":false},{"name":"DRONE_013.wav","sha256":"5548aac8a087f83faacd47da455af8a6e1f55489001200f222582ccdc81f7f08","bytes":1764052,"group_id":"DRONE_013","training_excluded":false},{"name":"DRONE_014.wav","sha256":"94ace9424b53e69c34ca8184ab3bac397996807047d4abf3a31abf96abc22d9b","bytes":1764052,"group_id":"DRONE_014","training_excluded":false},{"name":"DRONE_015.wav","sha256":"154de713d1e6bab713d91dedf66c623b313eb8dd74b9560fdc6199abc4f1d03e","bytes":1764052,"group_id":"DRONE_015","training_excluded":false},{"name":"DRONE_016.wav","sha256":"c46e30be454f61aa0dad797daa4dddd4e747e893150eb0b99fc27ede5ee090da","bytes":1764052,"group_id":"DRONE_016","training_excluded":false},{"name":"DRONE_017.wav","sha256":"789304d5fecdf52bee3eaea5b2b56ce20e67875b7d0df0b57bd15a7bf488719d","bytes":1764052,"group_id":"DRONE_017","training_excluded":false},{"name":"DRONE_018.wav","sha256":"06c806bdbe8499a31976b9c63be89afddc35fc20ebceb5b7fef8d48105d22490","bytes":1764052,"group_id":"DRONE_018","training_excluded":false},{"name":"DRONE_019.wav","sha256":"732ed57b672f7d1c6edc36f08fedad116119b4dd0d267c7e33ad20af6693acd0","bytes":1764052,"group_id":"DRONE_019","training_excluded":false},{"name":"DRONE_020.wav","sha256":"68b73a293ca70e8791481492122cdc81db56c4967673c9f27d99844127bcb23d","bytes":1764052,"group_id":"DRONE_020","training_excluded":false},{"name":"DRONE_021.wav","sha256":"7aba84d6e9edfa6766d18cc481022162e0c9a09734a502a6ac0789e8e9ce88a6","bytes":1764052,"group_id":"DRONE_021","training_excluded":false},{"name":"DRONE_022.wav","sha256":"ddc68de1fdb7b5ec7477d897af55e1a3881991bbe1587c9d083844050e685df9","bytes":1764052,"group_id":"DRONE_022","training_excluded":false},{"name":"DRONE_023.wav","sha256":"596be4f0dd105f98bc2b9a8fc0bbad69ba88645130f6cf54f683799555be53bf","bytes":1764052,"group_id":"DRONE_023","training_excluded":false},{"name":"DRONE_024.wav","sha256":"719c315afc8b9aca1bc8428aeeed1306be1d8f381806e1c4ef89b2338f849b7d","bytes":1764052,"group_id":"DRONE_024","training_excluded":false},{"name":"DRONE_025.wav","sha256":"4450b89707d09138e4aaf49b84df9d3188b3f7c778ba9af1fb149dec7534900a","bytes":1764052,"group_id":"DRONE_025","training_excluded":false},{"name":"DRONE_026.wav","sha256":"a3fca5caa403979b87122f425db4c04fabd13c7d5505d2112c77d680870905a3","bytes":1764052,"group_id":"DRONE_026","training_excluded":false},{"name":"DRONE_027.wav","sha256":"de55752be0ac29995d82cce0f0f9cedaa38a9c4405b0e180c27fed8030903fe5","bytes":1764052,"group_id":"DRONE_027","training_excluded":false},{"name":"DRONE_028.wav","sha256":"cf401858d8fc05a969a2ad3389cfeffb9b824b96da344cc9216e37bce597f3bb","bytes":1764052,"group_id":"DRONE_028","training_excluded":false},{"name":"DRONE_029.wav","sha256":"ca05e5b073ecc4e6567677d1a760f32ed96198c9f77b53ea90f07ba59915383b","bytes":1764048,"group_id":"DRONE_029","training_excluded":false},{"name":"DRONE_030.wav","sha256":"9bcb1ee9e4e80086e238042a90929ea8c43a5549792f617e2669ef438dd50552","bytes":1764052,"group_id":"DRONE_030","training_excluded":false},{"name":"HELICOPTER_001.wav","sha256":"ff04901b686c1b5aee3db7a7fcef94df677fd2e8cd49e6a840dc09e025260e90","bytes":1764048,"group_id":"HELICOPTER_001","training_excluded":false},{"name":"HELICOPTER_002.wav","sha256":"e9dbf544c189c0a1d80f3d599501432ecbd84e03ce5e1536114c6417d7de30d9","bytes":1764048,"group_id":"HELICOPTER_002","training_excluded":false},{"name":"HELICOPTER_003.wav","sha256":"976b0afd62fe6045673d37e0ba585979212603061c27bf9d1de4b23049b8f6af","bytes":1764048,"group_id":"HELICOPTER_003","training_excluded":false},{"name":"HELICOPTER_004.wav","sha256":"bdb7524eef645970587e1e96eaf23374255fc1789522f4bb69a42effc017d18b","bytes":1764048,"group_id":"HELICOPTER_004","training_excluded":false},{"name":"HELICOPTER_005.wav","sha256":"43961c8304d0de1d4dfb0ead9d97de8949a2e0893fc5bb3098ef9ab9a5e23120","bytes":1764048,"group_id":"HELICOPTER_005","training_excluded":false},{"name":"HELICOPTER_006.wav","sha256":"6ee524cabbff355ba4490ca975e284725531b59d2d660ef27ea0279215b95b08","bytes":1764048,"group_id":"HELICOPTER_006","training_excluded":false},{"name":"HELICOPTER_007.wav","sha256":"84102b6c5918a672ba8fb12750b7da9fd2d3f9857034627312f9d3a845610efc","bytes":1764048,"group_id":"HELICOPTER_007","training_excluded":false},{"name":"HELICOPTER_008.wav","sha256":"900639c6f5b02bbb3b42698550672aee7cd9b1fa77e8e9a8d00a0b0c1ed2e835","bytes":1764048,"group_id":"HELICOPTER_008","training_excluded":false},{"name":"HELICOPTER_009.wav","sha256":"7a69c14724199307229fb9fad26738ea4161a2734a8139cc865759a98718cc52","bytes":1764048,"group_id":"HELICOPTER_009","training_excluded":false},{"name":"HELICOPTER_010.wav","sha256":"a65b32fa367acfa63ce071b7e0b34b14e9aa82e571c2a5f96ae3b0067f8caa12","bytes":1764048,"group_id":"HELICOPTER_010","training_excluded":false},{"name":"HELICOPTER_011.wav","sha256":"cd9effd2f2552f29e6ae40fe6bf5f61ec9ead9cab6d428fbe242e939e9e50324","bytes":1764048,"group_id":"HELICOPTER_011","training_excluded":false},{"name":"HELICOPTER_012.wav","sha256":"94b8278d0c2ce8157250c229219505a8e18a1796d6d608de7a928eb896c19edb","bytes":1764048,"group_id":"HELICOPTER_012","training_excluded":false},{"name":"HELICOPTER_013.wav","sha256":"f4353777e7cafed31deef733eac90c65b0cc346e27503133acca5923ace0110e","bytes":1764052,"group_id":"HELICOPTER_013","training_excluded":false},{"name":"HELICOPTER_014.wav","sha256":"f0dbb7525b3dcb9d5269da1ef79ab1aaa94de912309ac4974caaacc75c8e103c","bytes":1764052,"group_id":"HELICOPTER_014","training_excluded":false},{"name":"HELICOPTER_015.wav","sha256":"a87b44b5cd1768bc657be7633c7ada7d93277d18c7214cfdbcbeb12c659f1cf4","bytes":1764052,"group_id":"HELICOPTER_015","training_excluded":false},{"name":"HELICOPTER_016.wav","sha256":"c68034426c7f0a97e4e95ebb7c7076893423835710aec2d7467c8b33b3dcef63","bytes":1764052,"group_id":"HELICOPTER_016","training_excluded":false},{"name":"HELICOPTER_017.wav","sha256":"7a8dcbbece59a44d276dc6b967ddf5b27a87a596d783eab3481f79510829f882","bytes":1764052,"group_id":"HELICOPTER_017","training_excluded":false},{"name":"HELICOPTER_018.wav","sha256":"769efc0d59b2107abb5e798a63781c43a1e382305f063f66d4f7f14a378a83b3","bytes":1764052,"group_id":"HELICOPTER_018","training_excluded":false},{"name":"HELICOPTER_019.wav","sha256":"f06f4a129f1ffe7d6ed9547f595d86ecf427e2d43a06c41f243d6a3ad8496ad4","bytes":1764052,"group_id":"HELICOPTER_019","training_excluded":false},{"name":"HELICOPTER_020.wav","sha256":"dec81d6f4386afda08ddb0129a2afc8a3c34f8239f30ffe58ddad9cd5cc585e0","bytes":1764052,"group_id":"HELICOPTER_020","training_excluded":false},{"name":"HELICOPTER_021.wav","sha256":"8677954e4aa82fe7db2281af1c7f6dbb064e82bc3f3fc66a04ccf23e2e571c05","bytes":1764052,"group_id":"HELICOPTER_021","training_excluded":false},{"name":"HELICOPTER_022.wav","sha256":"ff1f8395286e5cac5a24115a3b804480e1e0ebbc2fb903bae8c377ca0bd90fe5","bytes":1764052,"group_id":"HELICOPTER_022","training_excluded":false},{"name":"HELICOPTER_023.wav","sha256":"811f1e939c894a6f406d724f51bd6b06ba9dd271f03df3306ac456fc46af13a0","bytes":1764052,"group_id":"HELICOPTER_023","training_excluded":false},{"name":"HELICOPTER_024.wav","sha256":"d4b54c14bc09b136c9d7ce7bbd59ac27c6d34675c10bd675c7013d8294fdb3ba","bytes":1764052,"group_id":"HELICOPTER_024","training_excluded":false},{"name":"HELICOPTER_025.wav","sha256":"a82b468ae60f483a673b19794c8c43aefcea2bf66f7fa37fc533c918eb8fcb3d","bytes":1764052,"group_id":"HELICOPTER_025","training_excluded":false},{"name":"HELICOPTER_026.wav","sha256":"95184efe7292c1e9535a3f11580df0ccb0fff4a8536011690a5e738a1c393856","bytes":1764052,"group_id":"HELICOPTER_026","training_excluded":false},{"name":"HELICOPTER_027.wav","sha256":"a9555b719d5401d39c6713fe6dbb121298c1fab2015b29258a0143c9779a6fc0","bytes":1764052,"group_id":"HELICOPTER_027","training_excluded":false},{"name":"HELICOPTER_028.wav","sha256":"b4637a283e3175d935b9444cf29e5d47b1dec0736b45d365f517d3a3dc60af80","bytes":1764052,"group_id":"HELICOPTER_028","training_excluded":false},{"name":"HELICOPTER_029.wav","sha256":"00d4cb5e960aa0d613f412892993befb1b96f1c391709f8577d1a76cd47eb963","bytes":1764052,"group_id":"HELICOPTER_029","training_excluded":false},{"name":"HELICOPTER_030.wav","sha256":"55ecba3904aa9cf9427b6811a1ac69ae36e8c77c29c068742c5f971baf57f9cf","bytes":1764052,"group_id":"HELICOPTER_030","training_excluded":false}],"fraunhofer_record":"https://zenodo.org/api/records/17525811","dads_card":"https://huggingface.co/datasets/geronimobasso/drone-audio-detection-samples"}''')
SEED = 20261111
N_PER_CLASS = 8  # muestra de 24 de los 90 WAV inventariados (~42 MB); máx. 30 tras revisar coste y privacidad.
BASE = Path("/content/atalaya_piloto")
BASE.mkdir(parents=True, exist_ok=True)
COMMIT = PILOT["source_commit"]
print("Fuente fijada:", COMMIT, "| clips inventariados:", len(PILOT["clips"]),
      "| muestra prevista:", 3 * N_PER_CLASS, "| grupos:", PILOT["group_status"])


In [ ]:
# Partición por grupos, umbral elegido en validación y métricas con incertidumbre.
# Autocontenido (stdlib, numpy, scikit-learn): build_colab.py lo incrusta tal cual en el
# cuaderno de Colab, de modo que el código probado es el que se ejecuta allí.
import math
import random
from collections.abc import Sequence

import numpy as np
from sklearn.metrics import average_precision_score

PROVISIONAL_LABEL = "prueba técnica, contaminación entre particiones no descartada"


def split_indices_by_group(
    labels: Sequence[int], groups: Sequence[str], *, seed: int
) -> tuple[tuple[int, ...], tuple[int, ...], tuple[int, ...]]:
    """Train/validación/test por grupos: ningún grupo aparece en dos particiones."""
    if len(labels) != len(groups):
        raise ValueError("labels y groups deben tener la misma longitud")
    unique = sorted(set(groups))
    if len(unique) < 6:
        raise ValueError("se necesitan al menos seis grupos independientes")
    n_test = max(2, round(len(unique) * 0.2))
    n_validation = max(2, round(len(unique) * 0.2))
    if len(unique) - n_test - n_validation < 2:
        raise ValueError("grupos insuficientes para train/validation/test")
    for attempt in range(1000):
        shuffled = unique.copy()
        random.Random(seed + attempt).shuffle(shuffled)
        chosen = (
            set(shuffled[n_test + n_validation :]),
            set(shuffled[n_test : n_test + n_validation]),
            set(shuffled[:n_test]),
        )
        parts = tuple(
            tuple(i for i, group in enumerate(groups) if group in selected) for selected in chosen
        )
        if all({labels[i] for i in part} == {0, 1} for part in parts):
            train, validation, test = parts
            seen = [{groups[i] for i in part} for part in parts]
            if seen[0] & seen[1] or seen[0] & seen[2] or seen[1] & seen[2]:
                raise ValueError("un grupo aparece en varias particiones")
            return train, validation, test
    raise ValueError("no hay partición por grupos con ambas clases en los tres conjuntos")


def choose_threshold(y_validation: np.ndarray, p_validation: np.ndarray) -> float:
    """Umbral que maximiza F1 en validación; en empate, el más alto (menos alarmas)."""
    y = np.asarray(y_validation).astype(int)
    p = np.asarray(p_validation, dtype=float)
    if set(np.unique(y).tolist()) != {0, 1}:
        raise ValueError("la validación necesita ambas clases para fijar el umbral")
    best_threshold, best_f1 = math.nan, -1.0
    for threshold in np.unique(p):
        pred = p >= threshold
        tp = int(np.sum(pred & (y == 1)))
        fp = int(np.sum(pred & (y == 0)))
        fn = int(np.sum(~pred & (y == 1)))
        f1 = 2 * tp / (2 * tp + fp + fn) if tp else 0.0
        if f1 >= best_f1:
            best_threshold, best_f1 = float(threshold), f1
    return best_threshold


def binary_metrics(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    """Matriz de confusión, precision, recall y PR-AUC (precisión media)."""
    y = np.asarray(y_true).astype(int)
    p = np.asarray(scores, dtype=float)
    pred = p >= threshold
    tp = int(np.sum(pred & (y == 1)))
    fp = int(np.sum(pred & (y == 0)))
    fn = int(np.sum(~pred & (y == 1)))
    tn = int(np.sum(~pred & (y == 0)))
    both = set(np.unique(y).tolist()) == {0, 1}
    return {
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "precision": tp / (tp + fp) if tp + fp else math.nan,
        "recall": tp / (tp + fn) if tp + fn else math.nan,
        "pr_auc": float(average_precision_score(y, p)) if both else math.nan,
    }


def group_bootstrap(
    y_true: np.ndarray,
    scores: np.ndarray,
    groups: Sequence[str],
    threshold: float,
    *,
    n_boot: int = 2000,
    seed: int = 20261111,
    min_groups: int = 10,
) -> dict:
    """IC 95 % por percentiles remuestreando grupos completos, no clips sueltos."""
    y = np.asarray(y_true).astype(int)
    p = np.asarray(scores, dtype=float)
    unique = sorted(set(groups))
    if len(unique) < min_groups:
        return {"status": "INSUFICIENTE", "n_groups": len(unique), "min_groups": min_groups}
    members = {g: np.array([i for i, h in enumerate(groups) if h == g]) for g in unique}
    rng = np.random.default_rng(seed)
    samples: dict[str, list[float]] = {"precision": [], "recall": [], "pr_auc": []}
    for _ in range(n_boot):
        drawn = rng.choice(len(unique), size=len(unique), replace=True)
        idx = np.concatenate([members[unique[k]] for k in drawn])
        metrics = binary_metrics(y[idx], p[idx], threshold)
        for name in samples:
            samples[name].append(metrics[name])
    intervals = {}
    for name, values in samples.items():
        finite = np.asarray(values, dtype=float)
        finite = finite[np.isfinite(finite)]
        if finite.size < n_boot // 2:
            return {"status": "INSUFICIENTE", "n_groups": len(unique), "min_groups": min_groups}
        intervals[name] = (
            float(np.percentile(finite, 2.5)),
            float(np.percentile(finite, 97.5)),
        )
    return {"status": "OK", "n_groups": len(unique), "n_boot": n_boot, "intervals": intervals}


def false_alarms_per_hour(
    n_false_alarms: int, *, continuous_negative_seconds: float, min_hours: float
) -> dict:
    """Solo hay cifra con duración negativa continua conocida y suficiente."""
    hours = continuous_negative_seconds / 3600
    if hours < min_hours:
        return {"status": "NO VERIFICADA", "value": None, "hours": hours, "min_hours": min_hours}
    return {"status": "OK", "value": n_false_alarms / hours, "hours": hours, "min_hours": min_hours}


def metric_caption(*, provisional: bool) -> str:
    """Rótulo que acompaña a cada métrica según la fiabilidad de los grupos."""
    if provisional:
        return f"[{PROVISIONAL_LABEL}]"
    return "[grupos verificados; generalización limitada a estos grupos]"


In [ ]:
# Verifica la licencia original antes de descargar datos. Las redirecciones son restringidas.
ALLOWED_HOSTS = {"raw.githubusercontent.com", "api.github.com"}
def read_https(url, allowed_hosts, timeout=90):
    parsed = urlparse(url)
    assert parsed.scheme == "https" and parsed.hostname in allowed_hosts, "Origen no permitido"
    with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "Atalaya-Colab-Pilot/1"}), timeout=timeout) as r:
        redirected = urlparse(r.geturl())
        assert redirected.scheme == "https" and redirected.hostname in allowed_hosts, "Redirección no permitida"
        return r.read()

def read_source(url):
    return read_https(url, ALLOWED_HOSTS)

ROOT_URL = f"https://raw.githubusercontent.com/DroneDetectionThesis/Drone-detection-dataset/{COMMIT}"
license_bytes = read_source(ROOT_URL + "/LICENSE")
assert hashlib.sha256(license_bytes).hexdigest() == PILOT["license_sha256"]
assert b"CC0 1.0 Universal" in license_bytes
print("Licencia CC0 original verificada por SHA-256.")


In [ ]:
# Selección reproducible y descarga con SHA-256 por archivo. Solo datos públicos.
# La atribución de cada WAV del artículo todavía no está probada. La descarga
# exploratoria inicial ya se verificó; las ejecuciones futuras quedan cerradas.
# Decisión del usuario 2026-11-11 («Pero te lo autorizo, estoy despierto»): SOLO prueba técnica aislada
# en Colab; los WAV y el modelo nunca entran en el producto. El manifiesto del repo sigue sin aprobar.
SOURCE_INTAKE_DOWNLOAD_APPROVED = True
assert SOURCE_INTAKE_DOWNLOAD_APPROVED, "Procedencia por WAV sin verificar: no repetir la ingesta."
assert 1 <= N_PER_CLASS <= 30
randomizer = random.Random(SEED)
selected = []
for prefix in ("BACKGROUND", "HELICOPTER", "DRONE"):
    candidates = sorted((c for c in PILOT["clips"] if c["name"].startswith(prefix + "_")),
                        key=lambda c: c["name"])
    assert len(candidates) == 30
    candidates = [c for c in candidates if not c["training_excluded"]]
    # La fuente contiene al menos un duplicado exacto: nunca meterlo dos veces.
    unique = {c["sha256"]: c for c in candidates}
    assert len(unique) >= N_PER_CLASS
    selected += randomizer.sample(sorted(unique.values(), key=lambda c: c["name"]), N_PER_CLASS)

for item in selected:
    name = item["name"]
    assert name.replace("_", "").replace(".", "").isalnum() and name.endswith(".wav")
    path = BASE / name
    if path.exists():
        raw = path.read_bytes()
    else:
        raw = read_source(f"{ROOT_URL}/Data/Audio/{name}")
    assert len(raw) == item["bytes"] and hashlib.sha256(raw).hexdigest() == item["sha256"], name
    if not path.exists():
        path.write_bytes(raw)
    with wave.open(str(path), "rb") as wav:
        assert wav.getframerate() == 44100 and wav.getsampwidth() == 2
        assert wav.getnchannels() in (1, 2) and 9.9 <= wav.getnframes()/44100 <= 10.1
print(f"Descargados y verificados {len(selected)} WAV ({sum(c['bytes'] for c in selected)/1e6:.1f} MB).")


## Revisión de privacidad

Escucha al menos una muestra aleatoria de cada clase y comprueba que no haya voz
identificable u otros datos personales. El resultado automático de YAMNet más abajo
sirve de ayuda, **no sustituye** esta escucha. Si aparece voz, no entrenes; documenta
el clip y resuelve su tratamiento antes de seguir. Los WAV permanecen solo en el
almacenamiento temporal de este entorno de Colab.


In [ ]:
# Reproducción de tres clips públicos, uno por clase, para revisión humana.
review_names = []
for prefix in ("BACKGROUND", "HELICOPTER", "DRONE"):
    candidate = next(c for c in selected if c["name"].startswith(prefix + "_"))
    review_names.append(candidate["name"])
    audio, sr = librosa.load(BASE / candidate["name"], sr=None, mono=True)
    print(candidate["name"], f"{len(audio)/sr:.1f} s")
    display(Audio(audio, rate=sr))


In [ ]:
# YAMNet oficial: señal mono float32 a 16 kHz; embeddings congelados por ventana.
MODEL_URL = "https://tfhub.dev/google/yamnet/1"
yamnet = hub.load(MODEL_URL)
class_map_path = yamnet.class_map_path().numpy().decode()
class_names = pd.read_csv(class_map_path)["display_name"].tolist()
assert len(class_names) == 521

rows, vectors, frame_vectors, waveforms = [], [], {}, {}
for item in selected:
    name = item["name"]
    signal, sr = librosa.load(BASE / name, sr=16000, mono=True)
    assert sr == 16000 and np.isfinite(signal).all()
    scores, embeddings, _ = yamnet(tf.convert_to_tensor(signal, dtype=tf.float32))
    score_array, embedding_array = scores.numpy(), embeddings.numpy()
    assert embedding_array.shape[1] == 1024
    vectors.append(embedding_array.mean(axis=0))
    frame_vectors[name] = embedding_array
    waveforms[name] = signal
    speech_columns = [i for i, label in enumerate(class_names)
                      if label in {"Speech", "Conversation", "Narration, monologue", "Child speech, kid speaking"}]
    speech_score = float(score_array[:, speech_columns].max()) if speech_columns else float("nan")
    rows.append({"clip": name, "clase": name.split("_")[0], "dron": int(name.startswith("DRONE_")),
                 "ventanas": len(score_array), "max_speech_score": round(speech_score, 3),
                 "evento_YAMNet": class_names[int(score_array.mean(axis=0).argmax())]})
df = pd.DataFrame(rows)
display(df.sort_values("max_speech_score", ascending=False).head(10))
print("Si YAMNet marca voz, revisa esos clips y no uses material con datos personales sin base resuelta.")


In [ ]:
# Intake pendiente: la ficha CC0 no identifica el origen de cada WAV.
# Esta celda documenta el bloqueo; no se desbloquea cambiando una casilla.
# Decisión del usuario 2026-11-11: entrenamiento solo como prueba técnica; revisión de voz automática con
# la puntuación de voz de YAMNet (no sustituye a una escucha humana). Generalización: NO evaluada.
TRAINING_ALLOWED = True
assert TRAINING_ALLOWED, "Intake sin aprobar: derechos por fichero, revisión de voz, data card y agrupación."


In [ ]:
# Clasificador simple: regresión logística sobre la MEDIA de embeddings YAMNet por clip.
# Partición train/validación/test por group_id (ningún grupo en dos particiones).
# El umbral se fija en validación y se guarda; test solo se usa para medir.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay

assert TRAINING_ALLOWED, "Entrenamiento bloqueado: intake de derechos, privacidad y grupos pendiente."

PROVISIONAL = PILOT["group_status"] != "verified"
CAPTION = metric_caption(provisional=PROVISIONAL)
GROUP_OF = {c["name"]: c["group_id"] for c in PILOT["clips"]}
X = np.asarray(vectors)  # una fila por clip: media temporal de los embeddings de 1024 dimensiones
y = df["dron"].to_numpy().astype(int)
groups = [GROUP_OF[n] for n in df["clip"]]
train_idx, validation_idx, test_idx = (np.asarray(part) for part in
                                       split_indices_by_group(y.tolist(), groups, seed=SEED))
part_groups = [{groups[i] for i in part} for part in (train_idx, validation_idx, test_idx)]
assert not (part_groups[0] & part_groups[1] or part_groups[0] & part_groups[2]
            or part_groups[1] & part_groups[2]), "un grupo aparece en dos particiones"

head = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
head.fit(X[train_idx], y[train_idx])
threshold = choose_threshold(y[validation_idx], head.predict_proba(X[validation_idx])[:, 1])
MODEL_RECORD = {
    "modelo": "LogisticRegression sobre media de embeddings YAMNet por clip",
    "yamnet": MODEL_URL, "seed": SEED, "group_status": PILOT["group_status"],
    "umbral": threshold, "umbral_elegido_en": "validación (máximo F1; empate -> umbral mayor)",
    "particiones": {k: sorted(df["clip"].iloc[v]) for k, v in
                    (("train", train_idx), ("validacion", validation_idx), ("test", test_idx))},
    "coef_sha256": hashlib.sha256(head.coef_.tobytes() + head.intercept_.tobytes()).hexdigest(),
}
(BASE / "modelo_piloto.json").write_text(json.dumps(MODEL_RECORD, ensure_ascii=False, indent=1))
print("Umbral fijado en validación y guardado en modelo_piloto.json:", round(threshold, 4))
print("clips train/val/test:", len(train_idx), len(validation_idx), len(test_idx),
      "| grupos:", [len(g) for g in part_groups], CAPTION)

proba = head.predict_proba(X[test_idx])[:, 1]
m = binary_metrics(y[test_idx], proba, threshold)
boot = group_bootstrap(y[test_idx], proba, [groups[i] for i in test_idx], threshold, seed=SEED)
def interval(name):
    if boot["status"] != "OK":
        return (f"IC 95 % por grupo: NO DISPONIBLE "
                f"({boot['n_groups']} grupos de test < {boot['min_groups']})")
    lo, hi = boot["intervals"][name]
    return f"IC 95 % por grupo [{lo:.3f}, {hi:.3f}]"
print("Matriz de confusión (test):", {k: m[k] for k in ("tn", "fp", "fn", "tp")}, CAPTION)
for name in ("precision", "recall", "pr_auc"):
    print(f"{name}: {m[name]:.3f} · {interval(name)} {CAPTION}")
# Los clips son de ~10 s y no contiguos: no hay duración negativa continua conocida.
# Mínimo exploratorio de este cuaderno, no requisito (REQ-NFR-004 se evalúa en evals/frozen).
FA_MIN_CONTINUOUS_HOURS = 24.0
LONGEST_CONTINUOUS_NEGATIVE_S = 10.0
fa = false_alarms_per_hour(m["fp"], continuous_negative_seconds=LONGEST_CONTINUOUS_NEGATIVE_S,
                           min_hours=FA_MIN_CONTINUOUS_HOURS)
print("Falsas alarmas por hora:", fa["status"] if fa["value"] is None else f"{fa['value']:.3f}",
      f"(negativo continuo conocido {fa['hours']*3600:.0f} s < {FA_MIN_CONTINUOUS_HOURS:.0f} h)"
      if fa["value"] is None else "", CAPTION)
ConfusionMatrixDisplay.from_predictions(y[test_idx], (proba >= threshold).astype(int),
                                        labels=[0, 1], display_labels=["no dron", "dron"])
plt.title(f"Test · umbral de validación {threshold:.3f}\n{CAPTION}", fontsize=9)
plt.show()


In [ ]:
# Ilustración sobre un clip de test: espectrograma y puntuación por ventana.
# AVISO: el clasificador se entrenó con la MEDIA de embeddings por clip. Aplicarlo a ventanas
# sueltas de 0,96 s (salto 0,48 s) queda fuera de su distribución de entrenamiento: la curva
# NO está calibrada y el umbral (fijado por clip en validación) no es un umbral por ventana.
chosen = next((i for i in test_idx if y[i] == 1), test_idx[0])
name = df.iloc[chosen]["clip"]
signal = waveforms[name]
mel = librosa.feature.melspectrogram(y=signal, sr=16000, n_mels=64)
db = librosa.power_to_db(mel, ref=np.max)
window_scores = head.predict_proba(frame_vectors[name])[:, 1]
fig, (a, b) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
librosa.display.specshow(db, sr=16000, x_axis="time", y_axis="mel", ax=a)
a.set_title(f"{name} · espectrograma")
b.plot(np.arange(len(window_scores)) * 0.48, window_scores)
b.axhline(threshold, color="red", linestyle="--",
          label="umbral por clip (validación); no válido por ventana")
b.set(ylim=(0, 1), ylabel="Puntuación sin calibrar", xlabel="Tiempo (s)")
b.legend()
fig.suptitle("Modelo por clip (media de embeddings) aplicado a ventanas: "
             "curva NO calibrada, solo ilustrativa\n" + CAPTION, fontsize=9)
plt.tight_layout(); plt.show()


## Fuentes para las siguientes pruebas

- **Fraunhofer IDMT:** [registro original de Zenodo](https://zenodo.org/records/17525811),
  licencia CC BY 4.0; 1,515,112,211 bytes de audio multicanal. Su estructura y
  coordenadas deben validarse antes de estimar dirección de llegada (azimut). Esta
  descarga grande queda fuera del ensayo rápido de clasificación.
- **DroneDetectionThesis:** el ensayo anterior incluye helicópteros reales como
  confusor. Su conjunto es pequeño y carece de grupos fiables de vuelo/sitio por WAV.
- **DADS:** la [ficha del autor](https://huggingface.co/datasets/geronimobasso/drone-audio-detection-samples)
  declara MIT pero cita [Al-Emadi](https://github.com/saraalemadi/DroneAudioDataset/blob/master/LICENSE.md),
  que prohíbe usos de seguridad y defensa, y [UrbanSound8K](https://urbansounddataset.weebly.com/),
  que limita a usos no comerciales. Hasta tener procedencia por archivo y derechos compatibles, `DADS_ENABLED=False`;
  ningún clip suyo se descarga ni se usa para entrenar este cuaderno.


In [ ]:
# Auditoría de metadatos de Fraunhofer; NO descarga su ZIP de 1.5 GB.
# Misma restricción de hosts y redirecciones que la celda de licencia de Svanström.
FRAUNHOFER_HOSTS = {"zenodo.org"}
FRAUNHOFER = PILOT["fraunhofer_record"]
meta_record = json.loads(read_https(FRAUNHOFER, FRAUNHOFER_HOSTS, timeout=30))
assert meta_record["metadata"]["license"]["id"] == "cc-by-4.0"
files = meta_record["files"]
assert len(files) == 1 and files[0]["key"] == "MicArrayData30s.zip"
assert files[0]["size"] == 1515112211
print("Fraunhofer:", meta_record["metadata"]["title"])
print("Licencia:", meta_record["metadata"]["license"]["id"], "| archivo:", files[0]["key"])
print("DOA/azimut: fase posterior, con control de geometría y referencia GPS.")
DADS_ENABLED = False
assert not DADS_ENABLED
print("DADS bloqueado hasta resolver licencias y procedencia por archivo.")


**Lectura de resultados:** cualquier acierto o error anterior describe solo
estos clips y este reparto. No se ha medido tasa de falsas alarmas por hora,
distancia de detección, precisión en cárceles, nuevos drones o emplazamientos.
Para eso se necesitan grabaciones de campo independientes, negativos continuos,
revisión de privacidad y una evaluación congelada por vuelo/sitio.
